# Notebook 6 — Trend Analysis & Forecasting
**Purpose:** Detect revenue trends (UP/FLAT/DOWN) and forecast next year's revenue.
Methods: Linear regression for trend, Holt-Winters for forecasting top 20 companies.

> ⚠️ **Disclaimer:** All forecasts are model estimates, not financial advice.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, sqlite3, os, sys
from scipy import stats
sys.path.insert(0, os.path.abspath('..'))
sns.set_theme(style='darkgrid'); plt.rcParams['figure.figsize'] = (14, 6)

conn = sqlite3.connect(os.path.join('..', 'db.sqlite3'))
companies = pd.read_sql('SELECT c.*, s.sector_name FROM dim_company c LEFT JOIN dim_sector s ON c.sector_id=s.sector_id', conn)
years = pd.read_sql('SELECT * FROM dim_year ORDER BY sort_order', conn)
pl = pd.read_sql('SELECT * FROM fact_profit_loss', conn).merge(years[['year_id','year_label','sort_order','fiscal_year']], on='year_id')
print(f'P&L rows: {len(pl)}, Companies: {pl.company_id.nunique()}')

## Step 1: Linear Regression Trend Classification

In [ ]:
# Use last 5 years of data per company
max_order = pl[pl['sort_order'] < 9999]['sort_order'].max()
recent = pl[(pl['sort_order'] >= max_order - 5) & (pl['sort_order'] <= max_order)].copy()

trend_results = []
for symbol in recent['company_id'].unique():
    comp = recent[recent['company_id']==symbol].sort_values('sort_order')
    sales = comp['sales'].dropna()
    profit = comp['net_profit'].dropna()
    if len(sales) < 3: continue
    
    # Linear regression on sales
    x = np.arange(len(sales))
    slope_s, intercept_s, r_s, p_s, se_s = stats.linregress(x, sales)
    
    # Linear regression on profit
    slope_p = 0
    if len(profit) >= 3:
        x_p = np.arange(len(profit))
        slope_p, _, r_p, _, _ = stats.linregress(x_p, profit)
    
    # Classify trend
    avg_sales = sales.mean()
    norm_slope = slope_s / avg_sales if avg_sales > 0 else 0
    if norm_slope > 0.05: trend = 'UP'
    elif norm_slope < -0.05: trend = 'DOWN'
    else: trend = 'FLAT'
    
    trend_results.append({
        'company_id': symbol, 'sales_slope': round(slope_s, 2),
        'norm_slope': round(norm_slope, 4), 'r_squared': round(r_s**2, 3),
        'profit_slope': round(slope_p, 2), 'trend': trend,
        'data_points': len(sales)
    })

trends = pd.DataFrame(trend_results)
print(trends['trend'].value_counts())
trends.sort_values('norm_slope', ascending=False).head(10)

## Step 2: Trend Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'UP':'#10B981', 'FLAT':'#F59E0B', 'DOWN':'#EF4444'}
trend_counts = trends['trend'].value_counts()
axes[0].bar(trend_counts.index, trend_counts.values, color=[colors[t] for t in trend_counts.index])
axes[0].set_title('Revenue Trend Classification', fontweight='bold')

axes[1].hist(trends['norm_slope'], bins=25, color='#6366F1', edgecolor='white')
axes[1].axvline(x=0.05, color='green', linestyle='--', label='UP threshold')
axes[1].axvline(x=-0.05, color='red', linestyle='--', label='DOWN threshold')
axes[1].set_title('Normalized Slope Distribution', fontweight='bold')
axes[1].legend()
plt.tight_layout(); plt.show()

## Step 3: Holt-Winters Forecasting (Top 20 Companies)

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# Get top 20 companies by latest revenue
latest_sales = recent.sort_values('sort_order').groupby('company_id')['sales'].last().nlargest(20)
top20 = latest_sales.index.tolist()

forecasts = []
fig, axes = plt.subplots(5, 4, figsize=(20, 20))
axes = axes.flatten()

for i, symbol in enumerate(top20):
    comp = pl[pl['company_id']==symbol].sort_values('sort_order')
    sales = comp['sales'].dropna().values
    years_list = comp.loc[comp['sales'].notna(), 'year_label'].values
    
    if len(sales) < 6:
        # Fallback to linear extrapolation
        x = np.arange(len(sales))
        slope, intercept, _, _, _ = stats.linregress(x, sales)
        forecast_val = intercept + slope * len(sales)
        ci_lower = forecast_val * 0.9
        ci_upper = forecast_val * 1.1
    else:
        try:
            model = ExponentialSmoothing(sales, trend='add', seasonal=None, damped_trend=True)
            fitted = model.fit(optimized=True)
            fc = fitted.forecast(1)
            forecast_val = fc[0]
            residuals = fitted.resid
            se = np.std(residuals)
            ci_lower = forecast_val - 1.96 * se
            ci_upper = forecast_val + 1.96 * se
        except Exception:
            x = np.arange(len(sales))
            slope, intercept, _, _, _ = stats.linregress(x, sales)
            forecast_val = intercept + slope * len(sales)
            ci_lower = forecast_val * 0.9
            ci_upper = forecast_val * 1.1
    
    forecasts.append({
        'company_id': symbol, 'forecast_revenue': round(forecast_val, 2),
        'ci_lower': round(ci_lower, 2), 'ci_upper': round(ci_upper, 2),
        'latest_revenue': round(sales[-1], 2)
    })
    
    # Plot
    ax = axes[i]
    ax.plot(range(len(sales)), sales, 'b-o', markersize=3, label='Actual')
    ax.plot(len(sales), forecast_val, 'r*', markersize=12, label='Forecast')
    ax.fill_between([len(sales)-0.5, len(sales)+0.5], ci_lower, ci_upper, alpha=0.2, color='red')
    ax.set_title(symbol, fontweight='bold', fontsize=10)
    ax.tick_params(labelsize=7)

plt.suptitle('Revenue Forecasts — Top 20 Companies\n⚠️ Model estimate, not financial advice', fontweight='bold', fontsize=14)
plt.tight_layout(); plt.show()

forecast_df = pd.DataFrame(forecasts)
forecast_df

## Step 4: Export Results

In [ ]:
# Merge trends and forecasts
trends_export = trends.merge(forecast_df, on='company_id', how='left')
trends_export.to_csv('../data/trend_forecasts.csv', index=False)
print(f'✅ Exported {len(trends_export)} trend/forecast records to data/trend_forecasts.csv')
print('\n⚠️ DISCLAIMER: All forecasts are statistical model estimates, NOT financial advice.')
conn.close()